#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Text Classification - Active Learning (Inglés)

# Carga & Exploración de Datos

In [1]:
#Importar Librerías
import pandas as pd
import numpy as np
import spacy

In [2]:
#Importar Datos
df_labelled = pd.read_csv('/content/active_learning_spam_labelled.csv')
df_unlabelled = pd.read_csv('/content/active_learning_spam_unlabelled.csv')

In [3]:
#Verificar Columnas
print("Labelled Dataset:", df_labelled.columns)
print("Unlabelled Dataset:", df_unlabelled.columns)

Labelled Dataset: Index(['category', 'Message'], dtype='object')
Unlabelled Dataset: Index(['Message'], dtype='object')


In [4]:
#Verificar Tamaño
print("Dimensionalidad Labelled Dataset:", df_labelled.shape)
print("Dimensionalidad Unlabelled Dataset:", df_unlabelled.shape)

Dimensionalidad Labelled Dataset: (500, 2)
Dimensionalidad Unlabelled Dataset: (100, 1)


# Representación Vectorial

In [5]:
#Descargar spaCy pipeline
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
#Cargar Pipeline
nlp = spacy.load("en_core_web_sm")

In [7]:
#Obtener Doc Word Embeddings (Labelled)
doc_vectors_labelled = list()
for tweet in df_labelled['Message']:
  doc = nlp(tweet)
  doc_vectors_labelled.append(doc.vector)

In [8]:
#Obtener Doc Word Embeddings (Unlabelled)
doc_vectors_unlabelled = list()
for tweet in df_unlabelled['Message']:
  doc = nlp(tweet)
  doc_vectors_unlabelled.append(doc.vector)

In [9]:
#Generar Matriz X
X_labelled = np.array(doc_vectors_labelled)
X_unlabelled = np.array(doc_vectors_unlabelled)
print("Dimensionalidad X Labelled:", X_labelled.shape)
print("Dimensionalidad X Unlabelled:", X_unlabelled.shape)

Dimensionalidad X Labelled: (500, 96)
Dimensionalidad X Unlabelled: (100, 96)


In [10]:
#Generar Vector y
y_labelled = np.array(df_labelled['category'])
print("Dimensionalidad y Labelled:", y_labelled.shape)

Dimensionalidad y Labelled: (500,)


# Ciclo Active Learning

In [11]:
#Instalar modAL
!pip install git+https://github.com/modAL-python/modAL.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.5 MB/s eta 0:00:00


In [12]:
#Importar Librerías
import modAL
from modAL.models import ActiveLearner
from modAL.uncertainty import uncertainty_sampling
from sklearn.linear_model import LogisticRegression

In [13]:
#Crear Objeto ActiveLearner
learner = ActiveLearner(
    estimator = LogisticRegression(),
    query_strategy=uncertainty_sampling,
    X_training=X_labelled, y_training=y_labelled
)

In [14]:
#Seleccionar Casos No Etiquetados
query_idx, query_inst = learner.query(X_unlabelled)
print("Ejemplo(s) - Índice(s):", query_idx)

Ejemplo(s) - Índice(s): [39]


In [15]:
#Revisar Desempeño ActiveLearner
learner.score(X=X_labelled, y=y_labelled)

0.97

In [16]:
#Procesar Nuevos Casos Etiquetados
df_new_labelled = pd.read_csv('/content/active_learning_spam_new_labelled.csv')
y_new_labelled = np.array(df_new_labelled['category'])

In [17]:
#Incoporar Nuevos Datos Etiquetados
learner.teach(X_unlabelled[query_idx], y_new_labelled)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [18]:
#Revisar Desempeño ActiveLearner
learner.score(X=X_labelled, y=y_labelled)

0.974